# DX 704 Week 5 Project - Search, Minimax, Dynamic Programming


<font color = 'red'> FINAL SCORE: 40/40 </font>

This week's project will test your understanding of this week's concepts by asking you to simulate various algorithms by hand.
You will apply search, minimax and dynamic programming concepts to solve a variety of small planning problems.

The full project description, a template notebook and supporting materials are available on GitHub: [Project 5 Materials](https://github.com/bu-cds-dx704/dx704-project-05).

## Example Code

This week's assignment does not involve any coding.

## Part 1: Searching vs Games

Consider the state space illustrated below.
Each terminal state state is marked with a reward for reaching that state.
Each non-terminal state has two possible actions represented by the two outgoing arrows to later (lower) states.
The only rewards are for reaching the terminal states, there are no diminishing returns (i.e. $\gamma=1$), and there is no randomness so actions may be freely chosen.

![](https://github.com/bu-cds-dx704/dx704-project-05/blob/main/part1.png?raw=true)

Solve for the value of each non-terminal state according to the following three scenarios.

1. **Search**: There is one agent that picks all actions with the goal of maximizing the final reward.

2. **Minimax**: There are two agents P1 and P2. P1 controls the actions for the 1st and 3rd rows (i.e. the states marked A and D-G) while P2 controls the actions for the 2nd and 4th rows (i.e. the states B-C and H-J). P1 seeks to maximize the final reward, and P2 seeks to minimize the final reward.

3. **Maximin**: P1 and P2 control the same states as before, but P1 seeks to minimize the final reward, and P2 seeks to maximize the final reward.


Save your results in a file "**values-1.tsv**" with the column state with label `A-J` and columns `search_value`, `minimax_value`, and `maximin_value` that respectively correspond to the three scenarios.

Hint: Print out the image above and compute the values by hand in a bottom up fashion.

Submit the file "values-1.tsv" in Gradescope.

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### 1. Search (Maximizing Agent)

In [14]:
# The value of a node is the maximum value of its children. 

H = max(9, -1)
I = max(20, 1)
J = max(3,2)

D = max(H,I)
E = max(J, 5)
F = max(1, -1)
G = max(1, 0)

B = max(D,E)
C = max(F,G)
A = max(B,C)

search_values = {'H': H, 'I': I, 'J': J, 'D': D, 'E': E, 'F': F, 'G': G, 'B': B, 'C': C, 'A': A}
# print(search_values)

values_1_df = pd.DataFrame(list(search_values.items()), columns = ['state','search_value'])
values_1_df

,state,search_value
0,H,9
1,I,20
2,J,3
3,D,20
4,E,5
5,F,1
6,G,1
7,B,20
8,C,1
9,A,20


### 2. Minimax (P1 Maximizes, P2 Minimizes)

In [15]:
# P1 controls the actions at levels 1 (A) and 3 (D-G), and 
# P2 controls the actions at levels 2 (B-C) and 4 (H-J). P1 seeks to maximize the reward, while P2 seeks to minimize it.

H = min(+9, -1)
I = min(+20, +1)
J = min(+3, +2)

D = max(H, I)
E = max(J, +5)
F = max(+1, -1)
G = max(+1, 0)

B = min(D, E)
C = min(F, G)
A = max(B, C)

minimax_values = {'H': H, 'I': I, 'J': J, 'D': D, 'E': E, 'F': F, 'G': G, 'B': B, 'C': C, 'A': A}
print(minimax_values)

values_1_df['minimax_value'] = values_1_df['state'].map(minimax_values)
values_1_df

{'H': -1, 'I': 1, 'J': 2, 'D': 1, 'E': 5, 'F': 1, 'G': 1, 'B': 1, 'C': 1, 'A': 1}


,state,search_value,minimax_value
0,H,9,-1
1,I,20,1
2,J,3,2
3,D,20,1
4,E,5,5
5,F,1,1
6,G,1,1
7,B,20,1
8,C,1,1
9,A,20,1


### 3. Maximin (P1 Minimizes, P2 Maximizes)

In [16]:
# P1 and P2 control the same states as before, but their goals are reversed: 
# P1 wants to minimize the reward, and P2 wants to maximize it.

H = max(+9, -1)
I = max(+20, +1)
J = max(+3, +2)
D = min(H, I)
E = min(J, +5)
F = min(+1, -1)
G = min(+1, 0)
B = max(D, E)
C = max(F, G)
A = min(B, C)

maximin_values = {'H': H, 'I': I, 'J': J, 'D': D, 'E': E, 'F': F, 'G': G, 'B': B, 'C': C, 'A': A}
print(maximin_values)

values_1_df['maximin_value'] = values_1_df['state'].map(maximin_values)
values_1_df

{'H': 9, 'I': 20, 'J': 3, 'D': 9, 'E': 3, 'F': -1, 'G': 0, 'B': 9, 'C': 0, 'A': 0}


,state,search_value,minimax_value,maximin_value
0,H,9,-1,9
1,I,20,1,20
2,J,3,2,3
3,D,20,1,9
4,E,5,5,3
5,F,1,1,-1
6,G,1,1,0
7,B,20,1,9
8,C,1,1,0
9,A,20,1,0


In [17]:
values_1_df.to_csv('values-1.tsv', index=False, sep='\t')

## Part 2: Picking Up Sticks

The state space illustrated below corresponds to a variation of the game [Nim](https://en.wikipedia.org/wiki/Nim).
States labeled with a prefix of "p1_" correspond to states where player P1 chooses the action while states labeled with a prefix of "p2_" correspond to states where player P2 chooses the action.
The number in the suffix is the number of "sticks" remaining.
The players take turns choosing actions, and each action corresponds to removing one or two sticks.
When there are no more sticks, the player who would have picked an action loses.


![](https://github.com/bu-cds-dx704/dx704-project-05/blob/main/part2.png?raw=true)

For example, from the state labeled "p1_1", there is one stick left, player P1 removes the last stick, and player P2 loses.
The loss for P2 is represented by a final reward of +1.
A loss for P1 is represented by a final reward of -1.
Player P1 tries to maximize the final reward, and player P2 tries to minimize the final reward.

Solve for the value of each of the non-terminal states.
Save the results in a file "values-2.tsv" with columns `state` and `value`.

Submit the file "values-2.tsv" in Gradescope.

In [18]:
# using the Minimax algorithm, working backward from the terminal states.

states = range(1,6)
players = ['p1', 'p2']
game_states = [f'{p}_{s}' for s in states for p in players]
print(game_states)

game_values = {}

game_values[game_states[0]] = 1  # p1_1
game_values[game_states[1]] = -1 # p2_1
game_values[game_states[2]] = max(-1, +1)  # p1_2 
game_values[game_states[3]] = min(+1, -1) # p2_2
game_values[game_states[4]] = max(-1, -1)  # p1_3
game_values[game_states[5]] = min(+1, +1) # p2_3
game_values[game_states[6]] = max(+1, -1)  # p1_4
game_values[game_states[7]] = min(-1, +1) # p2_4
game_values[game_states[8]] = max(-1, +1)  # p1_5
game_values[game_states[9]] = min(+1, -1) # p2_5

# for key,val in game_values.items():
#     print(f'{key}: {val}')

values_2_df = pd.DataFrame(list(game_values.items()), columns = ['state','value'])
values_2_df

['p1_1', 'p2_1', 'p1_2', 'p2_2', 'p1_3', 'p2_3', 'p1_4', 'p2_4', 'p1_5', 'p2_5']


,state,value
0,p1_1,1
1,p2_1,-1
2,p1_2,1
3,p2_2,-1
4,p1_3,-1
5,p2_3,1
6,p1_4,1
7,p2_4,-1
8,p1_5,1
9,p2_5,-1


In [19]:
values_2_df.to_csv('values-2.tsv', index=False, sep='\t')

In [20]:

# Recursive implementation of minimax with memoization
# results_2 = {}

# def minimax(player: str, num_sticks: int):
#     state = f'{player}_{num_sticks}'
#     # print(f'state: {state}')
#     # print(type(state))

#     if state in results_2:
#         return results_2[state]

#     if num_sticks == 0:
#         value = -1 if player == 'p1' else 1
#         results_2[state] = value
#         return value
    
#     opponent = 'p2' if player == 'p1' else 'p1'
#     move_1 = minimax(opponent, num_sticks - 1)
#     move_2 = minimax(opponent, num_sticks - 2) if num_sticks - 2 >= 0 else None

#     if player == 'p1':  # P1 maximizes
#         best_move = max(move_1, move_2) if move_2 is not None else move_1
#     else:  # P2 minimizes
#         best_move = min(move_1, move_2) if move_2 is not None else move_1

#     results_2[state] = best_move
#     print(f'storing: {state}: {best_move}')
#     return best_move

In [21]:
# for sticks in range(1, 6):
#     value_p1 = minimax('p1', sticks)
#     value_p2 = minimax('p2', sticks)
#     print(f"p1_{sticks}: {value_p1}")
#     print(f"p2_{sticks}: {value_p2}")

# results_2

## Part 3: Searching a Maze

Consider the following maze.

![](https://github.com/bu-cds-dx704/dx704-project-05/blob/main/part3.png?raw=true)

State C is a terminal state giving reward +100.
The remaining states have a reward of -1 when they are reached.
So moving to state F has a value of +99 due to the reward of -1 at state F and the optimal action of moving to state C for the reward of +100 afterwards.

Compute the values for states A-J and S and save them in a file "`values-3.tsv`" with columns `state` and `value`.

In [22]:
states = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'S']

C = 100  # terminal state with reward +100
F = 99 
J = -1 + F
I = -1 + J
H = -1 + I
G = -1 + I
E = -1 + H
D = -1 + G


B = -1 + D
S = -1 + B
A = -1 + S

rewards = [A, B, C, D, E, F, G, H, I, J, S] 

for reward in rewards:
    print(reward)

# zip states and rewards into a dictionary
state_values = dict(zip(states, rewards))
# print(state_values)
values_3_df = pd.DataFrame(list(state_values.items()), columns = ['state','value'])
values_3_df

92
94
100
95
95
99
96
96
97
98
93


,state,value
0,A,92
1,B,94
2,C,100
3,D,95
4,E,95
5,F,99
6,G,96
7,H,96
8,I,97
9,J,98


In [23]:
values_3_df.to_csv('values-3.tsv', index=False, sep='\t')

Submit "values-3.tsv" in Gradescope.

Goal: find the value of each state, which is the total reward of the optimal path from that state to the terminal state C. 
- This can be solved by working backward from the terminal state using a dynamic programming approach. 
- The value of a state is the reward of the current state plus the value of the next optimal state.

This is a deterministic maze, so best to use backward dynamic programming: each state's value is its reward plus the value of the optimal next state

## Part 4: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.